# Healthcare ETL Pipeline — Jupyter Notebook

**Teaching version** of `healthcareDataLoader.py`  
Runs the same medallion pipeline `CSV → Staging → Bronze → Silver → Gold (Views)` cell by cell.

---

### How to use this notebook
1. Run cells **in order from top to bottom** — each cell depends on the ones above it.
2. **Variables are shared across all cells** — `cfg`, `conn`, `batch_id`, `pre_flight`, etc. are all available after the cell that creates them.
3. To **resume from a specific phase** (e.g. after a Bronze failure), just re-run that phase cell and all cells below it — you do not need to reload the CSV.
4. To run a **dry-run** (validate CSV only, no DB writes), set `DRY_RUN = True` in the configuration cell.

### Pipeline steps
| Cell | Step |
|------|------|
| Cell 2 | Imports |
| Cell 3 | Constants — 92-column schema contract |
| Cell 4 | Section 1 — Configuration (`load_config`) |
| Cell 5 | Section 2 — Database connection (`get_connection`, `run_stored_procedure`) |
| Cell 6 | Section 3 — Pre-flight validation helpers |
| Cell 7 | Section 4 — Staging loader |
| Cell 8 | Section 5 — Orchestrator (`should_run`, `print_summary`, `main`) |
| Cell 9 | **▶ RUN PIPELINE** — set options and call `main()` |


## Cell 2 — Imports

In [ ]:
import configparser
import csv
import hashlib
import time
from dataclasses import dataclass, field
from pathlib import Path

import pandas as pd
import pyodbc

print('Imports loaded successfully')

## Cell 3 — Constants

`EXPECTED_COLUMNS` is the 92-column schema contract. Pre-flight validation checks the CSV header against this list before touching the database.

In [ ]:
EXPECTED_COLUMNS = [
    "patient_id", "birth_year", "age", "sex", "race_ethnicity", "state",
    "encounter_id", "icd10_code", "diagnosis_display", "diagnosis_category",
    "severity_index", "comorbidity_count", "has_diabetes", "has_hypertension",
    "has_chf", "sbp_mmhg", "dbp_mmhg", "heart_rate_bpm", "spo2_pct",
    "temperature_f", "bmi", "respiratory_rate", "hba1c_pct", "glucose_mg_dl",
    "creatinine_mg_dl", "wbc_10e3_ul", "nt_probnp_pg_ml",
    "medication_adherence_pdc", "readmission_30d_flag", "triage_timestamp",
    "admit_timestamp", "bed_request_time", "bed_assign_time", "unit_assigned",
    "bed_occupancy_pct", "cpt_code", "procedure_display", "or_start", "or_end",
    "actual_or_minutes", "or_turnover_minutes", "los_days", "discharge_timestamp",
    "safety_incident_flag", "incident_type", "incident_severity", "claim_id",
    "payer", "npi_billing", "drg_weight", "submitted_charge_usd",
    "allowed_amount_usd", "patient_responsibility_usd", "fraud_upcoding_flag",
    "fraud_duplicate_flag", "fraud_unbundling_flag", "nurse_emp_id", "nurse_role",
    "nurse_unit", "nurse_tenure_years", "nurse_fte", "surgeon_emp_id",
    "surgeon_specialty", "shift_hours", "patients_per_nurse_ratio",
    "overtime_hours", "burnout_exhaustion_mbi", "burnout_cynicism_mbi",
    "burnout_personal_accomplishment_mbi", # ++ ADDED: MBI third subscale
    "turnover_risk_index", "cahps_nurse_communication",
    "cahps_doctor_communication", "cahps_responsiveness", "cahps_pain_management",
    "cahps_discharge_info", "cahps_care_transition", "cahps_cleanliness",
    "cahps_quietness", "surgical_kit_id", "kit_name", "kit_unit_cost_usd",
    "kit_current_stock", "kit_reorder_point", "kit_lead_time_days",
    "kit_expiration_date", "stockout_risk_flag", "weekly_procedure_volume",
    "projected_demand_4wk", "days_of_supply", "hedis_hba1c_tested",
    "hedis_hba1c_poor_control", "pdsa_cycle_id", "himss_emram_stage",
]

print(f"Schema contract: {len(EXPECTED_COLUMNS)} columns defined")

In [ ]:
BASE_DIR = Path.cwd()
CONFIG_PATH = BASE_DIR / 'config.ini'
CSV_THOUSAND_ROWS = BASE_DIR.parent / '1MRecords_SDG/test_1k/encounters_1m.csv'
CSV_FIFTY_ROWS = BASE_DIR.parent / '1MRecords_SDG/test_50k/encounters_1m.csv'
CSV_MILLION_ROWS = BASE_DIR.parent / '1MRecords_SDG/output/encounters_1m.csv'

## Cell 4 — Section 1: Configuration

`load_config()` reads `config.ini` from disk and returns a `ConfigParser` object — a dictionary-like structure containing all pipeline settings. Nothing is hardcoded in the code itself.

In [ ]:
def load_config(config_path: Path = Path("config.ini")) -> configparser.ConfigParser:
    """Read and return the pipeline configuration from config.ini.

    Args:
        config_path: Path to config.ini. Defaults to the current directory.

    Returns:
        A ConfigParser object — works like a dictionary of settings.

    Raises:
        FileNotFoundError: If config.ini does not exist.
    """
    if not config_path.exists():
        raise FileNotFoundError(f"config.ini not found at: {config_path}")
    cfg = configparser.ConfigParser()
    cfg.read(config_path, encoding="utf-8")
    return cfg

In [ ]:
cfg = load_config(CONFIG_PATH)
# cfg['database'].get("connection_string")

## Cell 5 — Section 2: Database Connection

Two functions:
- `get_connection(cfg)` — opens one SQL Server connection that is reused for the entire pipeline run.
- `run_stored_procedure(conn, procedure, batch_id)` — calls any of the three pipeline stored procedures (Bronze, Silver, Gold) and reads back their row-count OUTPUT parameters.

In [ ]:
def get_connection(cfg: configparser.ConfigParser) -> pyodbc.Connection:
    """Open and return a SQL Server connection with retry logic.

    Reads the connection string from config.ini. Retries up to 3 times
    with a 5-second wait between attempts.

    Note: pyodbc's default encoding behaviour is left intact for SQL Server.
    The Microsoft ODBC Driver communicates via UTF-16LE for NVARCHAR columns
    internally. Overriding setencoding/setdecoding breaks that contract and
    causes ASCII bytes to be misread as UTF-16LE code points, producing the
    CJK gibberish characteristic of this bug.

    Args:
        cfg: ConfigParser loaded from config.ini.

    Returns:
        An active pyodbc.Connection.

    Raises:
        pyodbc.Error: If all 3 connection attempts fail.
    """
    db = cfg["database"]

    if "connection_string" in db:
        conn_str = db["connection_string"]
    else:
        conn_str = f"DSN={db['dsn']};Database={db['database']};"

    timeout = int(db.get("connection_timeout", 30))
    last_error = None

    for attempt in range(1, 4):
        try:
            conn = pyodbc.connect(conn_str, timeout=timeout, autocommit=False)
            return conn
        except pyodbc.Error as exc:
            last_error = exc
            if attempt < 3:
                print(f"  Connection attempt {attempt} failed. Retrying in 5s...")
                time.sleep(5)

    raise last_error

In [ ]:
conn = get_connection(cfg=cfg)
# cursor = conn.cursor()
# tables = cursor.tables(tableType="TABLE")
# for t in tables:
#     print(f"{t.table_schem}.{t.table_name}")

In [ ]:
def run_stored_procedure(
    conn: pyodbc.Connection,
    procedure: str,
    batch_id: str,
    timeout: int = 3600,
) -> dict:
    """Execute a pipeline stored procedure and return its row-count results.
    Dynamically handles the extra @rows_clean parameter in the Silver SP.
    """
    conn.timeout = timeout
    cursor = conn.cursor()
    #cursor.timeout = timeout

    # The Silver procedure contains a 5th output parameter (@rows_clean)
    if "silver" in procedure.lower():
        sql = f"""
            DECLARE @rows_loaded   INT = 0,
                    @rows_rejected INT = 0,
                    @rows_flagged  INT = 0,
                    @rows_clean    BIGINT = 0,
                    @duration_ms   INT = 0;
            EXEC {procedure}
                @batch_id      = ?,
                @rows_loaded   = @rows_loaded   OUTPUT,
                @rows_rejected = @rows_rejected OUTPUT,
                @rows_flagged  = @rows_flagged  OUTPUT,
                @rows_clean    = @rows_clean    OUTPUT,
                @duration_ms   = @duration_ms   OUTPUT;
            SELECT @rows_loaded, @rows_rejected, @rows_flagged, @duration_ms, @rows_clean;
        """
    else:
        sql = f"""
            DECLARE @rows_loaded   INT = 0,
                    @rows_rejected INT = 0,
                    @rows_flagged  INT = 0,
                    @duration_ms   INT = 0;
            EXEC {procedure}
                @batch_id      = ?,
                @rows_loaded   = @rows_loaded   OUTPUT,
                @rows_rejected = @rows_rejected OUTPUT,
                @rows_flagged  = @rows_flagged  OUTPUT,
                @duration_ms   = @duration_ms   OUTPUT;
            SELECT @rows_loaded, @rows_rejected, @rows_flagged, @duration_ms, 0;
        """
        
    cursor.execute(sql, batch_id)
    row = cursor.fetchone()
    conn.commit()

    return {
        "rows_loaded":   row[0] if row else 0,
        "rows_rejected": row[1] if row else 0,
        "rows_flagged":  row[2] if row else 0,
        "duration_ms":   row[3] if row else 0,
        "rows_clean":    row[4] if row else 0,
    }

In [ ]:
# procedure = 'gold.usp_createOrAlterViews'
# batchid = '20260625_182914'

In [ ]:
# run_stored_procedure(conn, procedure, batchid)

## Cell 6 — Section 3: Pre-Flight Validation

Four functions that validate the source CSV before any data touches the database:
- `compute_checksum(csv_path)` — SHA-256 fingerprint of the file.
- `validate_header(csv_path)` — checks the 92-column header contract.
- `run_pre_flight(csv_path, cfg)` — runs all checks in sequence and returns a `PreFlightResult`.

The `PreFlightResult` dataclass holds the outcome: `passed`, `checksum_sha256`, `file_row_count`, `warnings`, `errors`.

In [ ]:
@dataclass
class PreFlightResult:
    """Holds the outcome of all pre-flight checks.

    Attributes:
        passed:             True only if all checks pass — pipeline proceeds.
        checksum_sha256:    SHA-256 fingerprint of the source file.
        file_row_count:     Total data rows found in the file.
        warnings:           Non-fatal issues (e.g. row count deviation).
        errors:             Fatal issues that stop the pipeline.
    """
    passed: bool = False
    checksum_sha256: str = ""
    file_row_count: int = 0
    warnings: list = field(default_factory=list)
    errors: list = field(default_factory=list)


In [ ]:
def compute_checksum(csv_path: Path) -> str:
    """Compute the SHA-256 checksum of the CSV file in streaming 8 MB blocks.

    Streaming keeps memory flat regardless of file size — the file is
    never fully loaded into RAM.

    Args:
        csv_path: Path to the CSV file.

    Returns:
        64-character lowercase hex string.
    """
    sha = hashlib.sha256()
    with csv_path.open("rb") as fh:
        for block in iter(lambda: fh.read(8_388_608), b""):
            sha.update(block)
    return sha.hexdigest()


In [ ]:
# compute_checksum(CSV_FIFTY_ROWS)

In [ ]:
def validate_header(csv_path: Path) -> list[str]:
    """Read the CSV header row and verify it matches the 92-column contract.

    Args:
        csv_path: Path to the CSV file.

    Returns:
        List of error strings. An empty list means the header is valid.
    """
    with csv_path.open(encoding="utf-8", newline="") as fh:
        header = next(csv.reader(fh), [])

    errors = []

    # Check total column count first
    if len(header) != len(EXPECTED_COLUMNS):
        errors.append(
            f"Column count mismatch — expected {len(EXPECTED_COLUMNS)}, "
            f"got {len(header)}"
        )

    # Check each column name individually for clear error messages
    for i, (expected, actual) in enumerate(
        zip(EXPECTED_COLUMNS, header[:len(EXPECTED_COLUMNS)])
    ):
        if expected != actual:
            errors.append(f"Column {i}: expected '{expected}', got '{actual}'")

    return errors




In [ ]:
# validate_header(CSV_FIFTY_ROWS)

In [ ]:
def run_pre_flight(
    csv_path: Path,
    cfg: configparser.ConfigParser,
    force_reload: bool = False,
) -> PreFlightResult:
    """Run all five pre-flight checks and return the result.

    Checks run in order and stop at the first fatal error:
        1. File exists and is not empty.
        2. SHA-256 checksum computed (printed for audit reference).
        3. Header schema matches the 92-column contract.
        4. Full file scan for encoding errors (prints count to console).
        5. Row count compared to expected count from the actual file.

    Args:
        csv_path:     Path to encounters_1m.csv.
        cfg:          ConfigParser with [pipeline] settings.
        force_reload: If True, skip any duplicate-file warnings.

    Returns:
        PreFlightResult with passed=True only if all checks pass.
    """
    result = PreFlightResult()
    
    expected_rows = 0
    with csv_path.open("rb") as fh:
        for block in iter(lambda: fh.read(8_388_608), b""):
            expected_rows += block.count(b"\n")
    expected_rows -= 1 #subtract header row

    tolerance = float(cfg["pipeline"].get("row_count_tolerance", 0.001))
    chunk_size = int(cfg["pipeline"].get("chunk_size", 50_000))

    print("\n  Running pre-flight validation...")

    # ── Check 1: File exists ──────────────────────────────────────────────────
    if not csv_path.exists() or csv_path.stat().st_size == 0:
        result.errors.append(f"File not found or empty: {csv_path}")
        print(f"  [FAIL] {result.errors[-1]}")
        return result

    size_mb = csv_path.stat().st_size / 1_048_576
    print(f"  [OK]   File found: {csv_path.name} ({size_mb:.1f} MB)")

    # ── Check 2: Checksum ─────────────────────────────────────────────────────
    print("  [...]  Computing SHA-256 checksum...", end="", flush=True)
    result.checksum_sha256 = compute_checksum(csv_path)
    print(f"\r  [OK]   SHA-256: {result.checksum_sha256}")

    # ── Check 3: Header schema ────────────────────────────────────────────────
    header_errors = validate_header(csv_path)
    if header_errors:
        result.errors.extend(header_errors)
        for err in header_errors:
            print(f"  [FAIL] {err}")
        return result
    print(f"  [OK]   Header validated — {len(EXPECTED_COLUMNS)} columns confirmed")

    # ── Check 4: Encoding scan ────────────────────────────────────────────────
    # Read the file in chunks; count rows with encoding replacement characters
    total_rows = 0
    encoding_errors = 0

    for chunk_df in pd.read_csv(
        csv_path,
        chunksize=chunk_size,
        dtype=str,
        keep_default_na=False,
        encoding="utf-8",
        encoding_errors="replace",  # \ufffd marks bad bytes instead of crashing
        on_bad_lines="warn",
    ):
        for row in chunk_df.itertuples(index=False):
            raw = ",".join(str(v) for v in row)
            if "\ufffd" in raw:   # replacement character = encoding problem
                encoding_errors += 1
        total_rows += len(chunk_df)

    result.file_row_count = total_rows

    if encoding_errors:
        msg = f"{encoding_errors:,} rows with encoding errors (will be loaded as-is)"
        result.warnings.append(msg)
        print(f"  [WARN] {msg}")
    else:
        print(f"  [OK]   Encoding scan clean — {total_rows:,} rows scanned")

    # ── Check 5: Row count ────────────────────────────────────────────────────
    deviation = abs(total_rows - expected_rows) / expected_rows
    if deviation > tolerance:
        msg = (
            f"Row count {total_rows:,} deviates {deviation:.2%} "
            f"from expected {expected_rows:,}"
        )
        result.warnings.append(msg)
        print(f"  [WARN] {msg}")
    else:
        print(f"  [OK]   Row count confirmed: {total_rows:,}")

    result.passed = not result.errors
    return result



In [ ]:
# run_pre_flight(CSV_FIFTY_ROWS, cfg)

## Cell 7 — Section 4: Staging Loader

Two functions:
- `truncate_staging(conn)` — wipes `stg.encounters_raw` before each fresh load.
- `load_csv_to_staging(csv_path, batch_id, conn, chunk_size, total_rows)` — streams the CSV into Staging in 50,000-row chunks using `fast_executemany=True`. Shows a live progress bar that moves proportionally to the actual file size.

In [ ]:
def truncate_staging(conn: pyodbc.Connection) -> None:
    """Truncate stg.encounters_raw before loading a fresh batch.

    Staging is a transient layer — it is wiped on every pipeline run.
    Only the data schemas are affected; no log or quarantine tables exist.

    Args:
        conn: Active pyodbc connection.
    """
    conn.cursor().execute("TRUNCATE TABLE stg.encounters_raw")
    conn.commit()
    print("  [OK]   stg.encounters_raw truncated")


In [ ]:
# truncate_staging(conn)

In [ ]:
# # ── Shared batch identifier for all Cell 7 test calls ────────────────────────
# # Must match what main() will use — defined here so manual tests and main()
# # both stamp rows with the same token.
# batch_id = time.strftime("%Y%m%d_%H%M%S")
# print(f"  batch_id = {batch_id}")

In [ ]:
def load_csv_to_staging(
    csv_path: Path,
    batch_id: str,
    conn: pyodbc.Connection,
    chunk_size: int = 50_000,
    total_rows: int = 1_000_000,
) -> int:
    """Stream the CSV into stg.encounters_raw in fixed-size chunks."""
    col_list = ", ".join(EXPECTED_COLUMNS) + ", _load_batch_id"
    placeholders = ", ".join(["?"] * (len(EXPECTED_COLUMNS) + 1))
    insert_sql = (
        f"INSERT INTO stg.encounters_raw ({col_list}) VALUES ({placeholders})"
    )

    total_loaded = 0
    pipeline_start = time.perf_counter()
    cursor = conn.cursor()
    cursor.fast_executemany = True  

    print("\n  Loading CSV → stg.encounters_raw")

    for chunk_seq, chunk_df in enumerate(
        pd.read_csv(
            csv_path,
            chunksize=chunk_size,
            dtype=str,             
            keep_default_na=False, 
            encoding="utf-8",
        ),
        start=1,
    ):
        chunk_start = time.perf_counter()

        # Tuples are created instantly. Empty cells remain empty strings ("") 
        # avoiding the pyodbc None-type inference error.
        rows = [
            tuple(row) + (batch_id,)
            for row in chunk_df.itertuples(index=False)
        ]

        try:
            cursor.executemany(insert_sql, rows)
            conn.commit()

            total_loaded += len(rows)
            elapsed = time.perf_counter() - pipeline_start
            rate = total_loaded / max(elapsed, 0.001)
            duration_ms = int((time.perf_counter() - chunk_start) * 1000)

            pct = min(total_loaded / max(total_rows, 1), 1.0)
            bar = "█" * int(40 * pct) + "░" * (40 - int(40 * pct))
            eta = (total_rows - total_loaded) / max(rate, 1)
            print(
                f"\r  [{bar}] {pct:5.1%}  "
                f"{total_loaded:>9,} rows  "
                f"{rate:>8,.0f} rows/s  "
                f"chunk {chunk_seq} ({duration_ms:,}ms)  "
                f"ETA {eta:4.0f}s",
                end="",
                flush=True,
            )

        except pyodbc.Error as exc:
            # Fatal error raised instantly to prevent silent data loss
            print(f"\n  [FATAL] Chunk {chunk_seq} failed — {exc}")
            conn.rollback()
            raise RuntimeError(f"Pipeline aborted due to staging insert failure at chunk {chunk_seq}.") from exc

    elapsed_total = time.perf_counter() - pipeline_start
    print(f"\n  [OK]   Staging complete — {total_loaded:,} rows in {elapsed_total:.1f}s")
    return total_loaded

In [ ]:
# truncate_staging(conn)
# load_csv_to_staging(CSV_FIFTY_ROWS, batch_id, conn)

## Cell 8 — Section 5: Pipeline Orchestrator

Three functions:
- `should_run(phase, start_from)` — decides whether a phase should execute (used to resume from a specific phase).
- `print_summary(metrics)` — prints the execution summary table to the notebook output.
- `main(start_from, dry_run)` — the conductor. Calls every other function in the correct order.

> **Note:** Unlike the original Python script, `main()` here takes parameters directly — no command-line flags needed.

In [ ]:
PHASES = ("staging", "bronze", "silver", "gold")


def should_run(phase: str, start_from: str) -> bool:
    """Return True if this phase should execute given the --start-from flag.

    Lets the user resume from a specific phase without re-running earlier
    ones. For example, --start-from silver skips staging and bronze.

    Args:
        phase:      The phase to check ('staging', 'bronze', etc.).
        start_from: The phase the user wants to start from.
    """
    return PHASES.index(phase) >= PHASES.index(start_from)


In [ ]:
def print_summary(metrics: dict) -> None:
    """Print the execution summary to the console.

    Args:
        metrics: Dict of all pipeline metrics collected during the run.
    """
    total = metrics.get("total_rows_input", 0)
    clean = metrics.get("silver_rows_clean", 0)
    flagged = metrics.get("silver_rows_flagged", 0)
    dq_rate = round(clean / max(total, 1) * 100, 2)

    print(f"\n  {'═' * 58}")
    print(f"  EXECUTION SUMMARY")
    print(f"  {'─' * 58}")
    print(f"  Source rows (Staging)   : {total:>12,}")
    print(f"  Rows in Bronze          : {metrics.get('bronze_rows_loaded', 0):>12,}")
    print(f"  Rows rejected (Bronze)  : {metrics.get('bronze_rows_rejected', 0):>12,}")
    print(f"  Rows in Silver          : {metrics.get('silver_rows_loaded', 0):>12,}")
    print(f"  Rows flagged (Silver)   : {flagged:>12,}")
    print(f"  Clean rows (Silver)     : {clean:>12,}")
    print(f"  Data quality rate       : {dq_rate:>11.2f}%")
    print(f"  {'─' * 58}")
    print(f"  Phase — Staging         : {metrics.get('staging_seconds', 0):>10}s")
    print(f"  Phase — Bronze          : {metrics.get('bronze_seconds', 0):>10}s")
    print(f"  Phase — Silver          : {metrics.get('silver_seconds', 0):>10}s")
    print(f"  Phase — Gold            : {metrics.get('gold_seconds', 0):>10}s")
    print(f"  Total duration          : {metrics.get('total_seconds', 0):>10}s")
    print(f"  Status                  : {'COMPLETED':>12}")
    print(f"  {'═' * 58}\n")


In [ ]:
def main(csv_path: Path, conn: pyodbc.Connection, start_from: str = "staging", dry_run: bool = False) -> None:
    """Run the full ETL pipeline.

    In the notebook, call this function directly from the final cell.
    Use the parameters to control pipeline behaviour instead of
    command-line flags.

    Args:
        start_from: Phase to start from — 'staging' | 'bronze' | 'silver' | 'gold'.
                    Use this to resume after a failed run without reloading the CSV.
        dry_run:    If True, validate the CSV only and write nothing to the database.
    """

    # ── Step 1: Load configuration ────────────────────────────────────────────
    print("\n  Healthcare ETL Pipeline — Starting")
    print(f"  {'─' * 40}")

    try:
        cfg = load_config(CONFIG_PATH)
    except FileNotFoundError as exc:
        print(f"  [FAIL] {exc}")
        raise

    # csv_path = Path(cfg["paths"]["csv_file"])
    chunk_size = int(cfg["pipeline"]["chunk_size"])
    cmd_timeout = int(cfg["database"]["command_timeout"])

    print(f"  Source  : {csv_path}")
    print(f"  Start from phase: {start_from.upper()}")

    # ── Step 2: Connect to SQL Server ─────────────────────────────────────────
    print("\n  Connecting to SQL Server...")
    # try:
    #     conn = get_connection(cfg)
    #     print(f"  [OK]   Connected to {cfg['database']['database']}")
    # except pyodbc.Error as exc:
    #     print(f"  [FAIL] Connection failed — {exc}")
    #     raise

    # Use a simple timestamp string as the batch identifier
    batch_id = time.strftime("%Y%m%d_%H%M%S")
    metrics: dict = {}
    pipeline_start = time.perf_counter()

    try:
        # ── Step 3: Pre-flight validation ─────────────────────────────────────
        pre_flight = run_pre_flight(csv_path, cfg)

        if not pre_flight.passed:
            print(f"\n  [FAIL] Pre-flight failed — pipeline stopped.")
            raise RuntimeError("Pre-flight validation failed.")

        for warning in pre_flight.warnings:
            print(f"  [WARN] {warning}")

        if dry_run:
            print("\n  [OK]   Dry run complete — no data written to database.")
            return

        metrics["total_rows_input"] = pre_flight.file_row_count

        # ── Step 4: Staging ───────────────────────────────────────────────────
        if should_run("staging", start_from):
            t0 = time.perf_counter()
            truncate_staging(conn)
            staging_rows = load_csv_to_staging(
                csv_path, batch_id, conn, chunk_size,
                total_rows=pre_flight.file_row_count
            )
            metrics["staging_rows_loaded"] = staging_rows
            metrics["staging_seconds"] = int(time.perf_counter() - t0)

        # ── Step 5: Bronze ────────────────────────────────────────────────────
        if should_run("bronze", start_from):
            print("\n  Running bronze.usp_loadFromStaging...")
            t0 = time.perf_counter()
            bronze = run_stored_procedure(
                conn, "bronze.usp_loadFromStaging", batch_id, cmd_timeout
            )
            metrics["bronze_rows_loaded"] = bronze["rows_loaded"]
            metrics["bronze_rows_rejected"] = bronze["rows_rejected"]
            metrics["bronze_seconds"] = int(time.perf_counter() - t0)
            print(
                f"  [OK]   Bronze — "
                f"loaded: {bronze['rows_loaded']:,}  "
                f"rejected: {bronze['rows_rejected']:,}  "
                f"({metrics['bronze_seconds']}s)"
            )

        # ── Step 6: Silver ────────────────────────────────────────────────────
        if should_run("silver", start_from):
            print("\n  Running silver.usp_loadFromBronze...")
            t0 = time.perf_counter()
            silver = run_stored_procedure(
                conn, "silver.usp_loadFromBronze", batch_id, cmd_timeout
            )
            metrics["silver_rows_loaded"] = silver["rows_loaded"]
            metrics["silver_rows_flagged"] = silver["rows_flagged"]
            metrics["silver_rows_clean"] = (
                silver["rows_loaded"] - silver["rows_flagged"]
            )
            metrics["silver_seconds"] = int(time.perf_counter() - t0)
            print(
                f"  [OK]   Silver — "
                f"loaded: {silver['rows_loaded']:,}  "
                f"flagged: {silver['rows_flagged']:,}  "
                f"clean: {metrics['silver_rows_clean']:,}  "
                f"({metrics['silver_seconds']}s)"
            )

        # ── Step 7: Gold views ────────────────────────────────────────────────
        if should_run("gold", start_from):
            print("\n  Running gold.usp_createOrAlterViews...")
            t0 = time.perf_counter()
            run_stored_procedure(
                conn, "gold.usp_createOrAlterViews", batch_id, cmd_timeout
            )
            metrics["gold_seconds"] = int(time.perf_counter() - t0)
            print(f"  [OK]   Gold views created ({metrics['gold_seconds']}s)")

        # ── Step 8: Print summary ─────────────────────────────────────────────
        metrics["total_seconds"] = int(time.perf_counter() - pipeline_start)
        print_summary(metrics)

    except Exception as exc:
        print(f"\n  [FAIL] Unexpected error — {exc}")
        raise

## Cell 9 — ▶ Run the Pipeline

Set the two options below, then run this cell to execute the full pipeline.

**`START_FROM`** — which phase to begin from:
- `'staging'` — full run from the beginning *(default)*
- `'bronze'` — skip CSV loading, reprocess from Staging into Bronze
- `'silver'` — skip Staging and Bronze, reprocess Silver and Gold only
- `'gold'` — rebuild Gold views only (fastest, ~seconds)

**`DRY_RUN`** — set to `True` to validate the CSV without writing anything to the database.

In [ ]:
# ── Configure pipeline options here ──────────────────────────────────────────
START_FROM = 'gold'   # 'staging' | 'bronze' | 'silver' | 'gold'
DRY_RUN    = False       # True = validate CSV only, no DB writes

# ── Run ──────────────────────────────────────────────────────────────────────
main(CSV_MILLION_ROWS,conn, start_from=START_FROM, dry_run=DRY_RUN)